In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Initialize Spark session
spark = SparkSession.builder \
    .appName("scd_1_implementation") \
    .getOrCreate()



schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", IntegerType(), True)])


data_target = [    
    (1, "Yash", 52000),
    (2, "Mukesh", 62000),
    (3, "Ram", 72000),
    (4, "Krishna", 82000),

]

data_target_df = spark.createDataFrame(data_target, schema)

print("Target Table:")
data_target_df.show()

data_new = [    
    (1, "Yash", 52000), #no change
    (2, "Mukesh", 60000), #salary changed
    (3, "Charlie", 70000), #name changed
    (5, "Radha", 90000), #new record
]   
data_new_df = spark.createDataFrame(data_new, schema)

print("New Data:")
data_new_df.show()


#----------------------------------------------------------------------------------------------------
#changed
changed_condition = ((data_new_df.name != data_target_df.name) | \
            (data_new_df.salary != data_target_df.salary))

changed_records_df = data_new_df.join(data_target_df, on='emp_id', how='inner') \
    .where(changed_condition) \
    .select(data_new_df.emp_id, data_new_df.name, data_new_df.salary)

print("Changed Records:")
changed_records_df.show(truncate=False)
print("Count of Changed Records:", changed_records_df.count())
#----------------------------------------------------------------------------------------------------



#----------------------------------------------------------------------------------------------------
#new
new_records_df = data_new_df.join(data_target_df, on='emp_id', how='left_anti')

print("New Records:")
new_records_df.show(truncate=False)
#----------------------------------------------------------------------------------------------------

#----------------------------------------------------------------------------------------------------
#history
history_records_df = data_target_df.join(data_new_df, on='emp_id', how='left_anti')  

print("history Records:")
history_records_df.show(truncate=False)
#----------------------------------------------------------------------------------------------------


#----------------------------------------------------------------------------------------------------
#unchanged
unchanged_condition = ((data_new_df.name == data_target_df.name) & \
            (data_new_df.salary == data_target_df.salary))

unchanged_records_df = data_new_df.join(data_target_df, on='emp_id', how='inner') \
    .where(unchanged_condition) \
    .select(data_new_df.emp_id, data_new_df.name, data_new_df.salary)

print("Unchanged Records:")
unchanged_records_df.show(truncate=False)
#----------------------------------------------------------------------------------------------------



final_df = history_records_df.union(changed_records_df).union(new_records_df).union(unchanged_records_df).orderBy("emp_id")

print("Final DataFrame:")
final_df.show(truncate=False)

spark.stop()




Target Table:
+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|     1|   Yash| 52000|
|     2| Mukesh| 62000|
|     3|    Ram| 72000|
|     4|Krishna| 82000|
+------+-------+------+

New Data:
+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|     1|   Yash| 52000|
|     2| Mukesh| 60000|
|     3|Charlie| 70000|
|     5|  Radha| 90000|
+------+-------+------+

Changed Records:
+------+-------+------+
|emp_id|name   |salary|
+------+-------+------+
|2     |Mukesh |60000 |
|3     |Charlie|70000 |
+------+-------+------+

Count of Changed Records: 2
New Records:
+------+-----+------+
|emp_id|name |salary|
+------+-----+------+
|5     |Radha|90000 |
+------+-----+------+

history Records:
+------+-------+------+
|emp_id|name   |salary|
+------+-------+------+
|4     |Krishna|82000 |
+------+-------+------+

Unchanged Records:
+------+----+------+
|emp_id|name|salary|
+------+----+------+
|1     |Yash|52000 |
+------+----+------+

Final DataFra